Evaluating LLaMA 3.3 on Sindhi Matrix with English Embeddings
This notebook evaluates the performance of LLaMA 3.3 on a custom MCQ dataset consisting of Sindhi matrix sentences with English lexical embeddings. The dataset contains 100 multiple-choice questions across three domains: History, Culture, and Geography.

Each question follows a zero-shot prompting approach, the model is given no prior examples and must select the correct answer (A, B, C, or D) based solely on the question.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Sindhi Matrix with english options - Sheet1.csv to Sindhi Matrix with english options - Sheet1.csv


In [2]:
import pandas as pd

df = pd.read_csv("Sindhi Matrix with english options - Sheet1.csv", encoding="utf-8")

## Displaying RTL Text

Since the dataset contains Sindhi text (written right-to-left),
we use an HTML rendering function to display it correctly in the notebook.

In [3]:
from IPython.display import HTML, display

def show_rtl(text):
    display(HTML(f'<div dir="rtl">{text}</div>'))

In [4]:
show_rtl(df.iloc[3, 2])

## Dataset Preview

Here we preview the first few rows of the dataset to verify
it has loaded correctly and check the structure of the questions.


In [5]:
import pandas as pd
df = pd.read_csv("Sindhi Matrix with english options - Sheet1.csv")
df.head()

,BaseID,Domain,Sindhi Matrix,A,B,C,D,answer,Source
0,1.0,Geography,سنڌ ۾ سڀ کان وڏو river ڪهڙو آهي؟,Lyari River,Indus River,Malir River,Hub River,B,NaN
1,2.0,History,سنڌ جو name ڪهڙي درياهُ کان ورتو ويو آهي؟,Malir River,Lyari River,Sindhu River,Hub River,C,Asaan sindhi 9th and 10th sindh board
2,3.0,Culture,روايتي سنڌي اجرڪ ٺاهڻ جا ڪيترا steps آهن؟,12,10,14,15,C,Asaan sindhi 9th and 10th sindh board
3,4.0,History,عربن سنڌي اجرڪ کي ڪهڙو name ڏنو؟,Khaddar,Azraq,Muslin,Chiffon,B,Asaan sindhi 9th and 10th sindh board
4,5.0,Culture,"سنڌي اجرڪ جي حوالي سان لفظ ""ازرق"" جي ڇا meanin...",White color,Red color,Green color,Blue color,D,Asaan sindhi 9th and 10th sindh board


## Building the Zero-Shot Prompt

Each MCQ is converted into a structured prompt. The model is instructed
to respond with only a single letter (A, B, C, or D), this is the
zero-shot prompting strategy used throughout this study.

In [7]:
def build_prompt(row):
    return f"""
You are given a multiple-choice question. Select the correct answer (A, B, C, or D).

Question:
{row['Sindhi Matrix']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Respond with ONLY one letter: A, B, C, or D.
"""

In [8]:
build_prompt(df.iloc[0])

'\nYou are given a multiple-choice question. Select the correct answer (A, B, C, or D).\n\nQuestion:\nسنڌ ۾ سڀ کان وڏو river ڪهڙو آهي؟\n\nA. Lyari River\nB.  Indus River\nC. Malir River\nD. Hub River\n\nRespond with ONLY one letter: A, B, C, or D.\n'

In [9]:
print(build_prompt(df.iloc[0]))


You are given a multiple-choice question. Select the correct answer (A, B, C, or D).

Question:
سنڌ ۾ سڀ کان وڏو river ڪهڙو آهي؟

A. Lyari River
B.  Indus River
C. Malir River
D. Hub River

Respond with ONLY one letter: A, B, C, or D.



## Connecting to LLaMA 3.3 via Groq API

LLaMA 3.3 (70B) is accessed through the Groq API.
Temperature is set to 0 to ensure deterministic,
consistent responses across all questions.

In [10]:
!pip install groq

In [11]:
from groq import Groq

In [12]:
client = Groq(api_key="gsk***********************************")

In [14]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello in one word"}
    ]
)

print(response.choices[0].message.content)

Hello


In [15]:
row = df.iloc[0]

prompt = build_prompt(row)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print(response.choices[0].message.content)

B


## Running the Evaluation

Each question is passed to LLaMA 3.3 individually using zero-shot prompting.
The model's response is parsed to extract the answer letter.
Responses are then scored using a binary system:
- 1 = Correct
- 0 = Incorrect

In [16]:
import re

results = []

def extract_answer(text):
    match = re.search(r"[ABCD]", text)
    return match.group(0) if match else None

for i, row in df.iterrows():
    prompt = build_prompt(row)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    output = response.choices[0].message.content
    model_answer = extract_answer(output)

    correct = row["answer"]
    score = 1 if model_answer == correct else 0

    results.append(score)

    print(f"{i+1}/{len(df)} → {model_answer} | correct: {correct} | score: {score}")

1/100 → B | correct: B | score: 1
2/100 → C | correct: C | score: 1
3/100 → B | correct: C | score: 0
4/100 → B | correct: B | score: 1
5/100 → D | correct: D | score: 1
6/100 → B | correct: B | score: 1
7/100 → A | correct: A | score: 1
8/100 → B | correct: B | score: 1
9/100 → A | correct: A | score: 1
10/100 → A | correct: A | score: 1
11/100 → B | correct: B | score: 1
12/100 → B | correct: D | score: 0
13/100 → C | correct: C | score: 1
14/100 → A | correct: A | score: 1
15/100 → A | correct: D | score: 0
16/100 → B | correct: A | score: 0
17/100 → C | correct: C | score: 1
18/100 → B | correct: B | score: 1
19/100 → A | correct: A | score: 1
20/100 → D | correct: B | score: 0
21/100 → A | correct: A | score: 1
22/100 → C | correct: C | score: 1
23/100 → A | correct: D | score: 0
24/100 → A | correct: A | score: 1
25/100 → D | correct: A | score: 0
26/100 → B | correct: B | score: 1
27/100 → A | correct: D | score: 0
28/100 → B | correct: C | score: 0
29/100 → A | correct: A | sco

## Results

### Overall Accuracy
We calculate the overall accuracy across all 100 questions,
followed by a breakdown by domain (History, Culture, Geography)
to identify if performance varied across topic areas.

In [17]:
df["score"] = results

In [18]:
df.groupby("Domain")["score"].mean()

,score
Domain,
Culture,0.700000
Geography,0.609756
History,0.655172


In [19]:
df["score"].mean()

np.float64(0.65)

In [20]:
import pandas as pd

# convert score into readable labels
df["result"] = df["score"].apply(lambda x: "correct" if x == 1 else "wrong")

# count correct/wrong per domain (CAPITAL D FIX)
counts = df.groupby(["Domain", "result"]).size().unstack(fill_value=0)

print(counts)

result     correct  wrong
Domain                   
Culture         21      9
Geography       25     16
History         19     10
